In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
CATALOG = "workspace"
SCHEMA = "final"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_person"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_person"

In [0]:
person_df = spark.table(BRONZE_TABLE)

display(person_df)

In [0]:
print(
    f"Bronze Count : {person_df.count()}"
)

In [0]:
def clean_text(column_name):

    cleaned = trim(
        regexp_replace(
            regexp_replace(
                col(column_name),
                "\\+",
                ""
            ),
            "&",
            ""
        )
    )

    return when(
        cleaned == "",
        None
    ).otherwise(cleaned)

In [0]:
null_analysis_df = person_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in person_df.columns
])

display(null_analysis_df)

In [0]:
person_clean_df = (
    person_df

    .withColumn(
        "BusinessEntityID",
        clean_text("BusinessEntityID")
    )

    .withColumn(
        "PersonType",
        clean_text("PersonType")
    )

    .withColumn(
        "Title",
        clean_text("Title")
    )

    .withColumn(
        "FirstName",
        initcap(clean_text("FirstName"))
    )

    .withColumn(
        "MiddleName",
        initcap(clean_text("MiddleName"))
    )

    .withColumn(
        "LastName",
        initcap(clean_text("LastName"))
    )

    .withColumn(
        "Suffix",
        clean_text("Suffix")
    )

    .withColumn(
        "rowguid",
        clean_text("rowguid")
    )

    .withColumn(
        "ModifiedDate",
        clean_text("ModifiedDate")
    )
)

In [0]:
person_clean_df = (
    person_clean_df

    .withColumn(
        "BusinessEntityID",
        expr(
            "try_cast(BusinessEntityID as BIGINT)"
        )
    )

    .withColumn(
        "NameStyle",
        expr(
            "try_cast(NameStyle as INT)"
        )
    )

    .withColumn(
        "EmailPromotion",
        expr(
            "try_cast(EmailPromotion as INT)"
        )
    )
)

In [0]:
person_clean_df = (
    person_clean_df
    .withColumn(
        "ModifiedDate",
        expr(
            "try_cast(ModifiedDate as timestamp)"
        )
    )
)

In [0]:
person_valid_df = (
    person_clean_df
    .filter(col("BusinessEntityID").isNotNull())
    .filter(col("FirstName").isNotNull())
    .filter(col("LastName").isNotNull())
)

In [0]:
window_spec = (
    Window
    .partitionBy("BusinessEntityID")
    .orderBy(
        col("ingested_at").desc()
    )
)

person_dedup_df = (
    person_valid_df
    .withColumn(
        "rn",
        row_number().over(window_spec)
    )
    .filter(col("rn") == 1)
    .drop("rn")
)

In [0]:
dq_id = person_dedup_df.filter(
    col("BusinessEntityID").isNull()
).count()

dq_first = person_dedup_df.filter(
    col("FirstName").isNull()
).count()

dq_last = person_dedup_df.filter(
    col("LastName").isNull()
).count()

dq_dup = (
    person_dedup_df
    .groupBy("BusinessEntityID")
    .count()
    .filter(col("count") > 1)
    .count()
)

print("========== DQ SUMMARY ==========")
print(f"BusinessEntityID Nulls : {dq_id}")
print(f"FirstName Nulls        : {dq_first}")
print(f"LastName Nulls         : {dq_last}")
print(f"Duplicate IDs          : {dq_dup}")

In [0]:
person_dedup_df = (
    person_dedup_df
    .withColumn(
        "person_business_hash",
        sha2(
            concat_ws(
                "|",
                coalesce(col("FirstName"),lit("")),
                coalesce(col("MiddleName"),lit("")),
                coalesce(col("LastName"),lit("")),
                coalesce(col("Title"),lit(""))
            ),
            256
        )
    )
)

In [0]:
silver_person_df = (
    person_dedup_df
    .withColumn(
        "silver_load_timestamp",
        current_timestamp()
    )
    .withColumn(
        "silver_layer",
        lit("SILVER")
    )
)

In [0]:
(
    silver_person_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(SILVER_TABLE)
)

In [0]:
silver_df = spark.table(SILVER_TABLE)

print(
    f"Silver Count : {silver_df.count()}"
)

display(silver_df)